# Quant Data — what the API actually serves

The feed behind the golden-sweep signal, mapped end to end: how to reach it from
Python, every tool it exposes, the record it returns, and how far back it goes.

The point of this notebook is to answer one question — **with Quant Data for
signals, Alpaca for underlying history, and Robinhood for execution, what is still
missing before the sweep strategy can be honestly backtested?** The last section
answers it; everything before is the evidence.

Companion to [`research/sweep_events/`](../sweep_events/), which consumes this feed,
and to `papers/quantdata_trade_types.md` for the print-type taxonomy.

## 1. Reaching it from Python

Quant Data is registered as an **MCP server**, which normally means only Claude's
tool layer can call it. Two facts make it scriptable:

1. The transport is plain HTTPS carrying **JSON-RPC 2.0** — `tools/list` and
   `tools/call` are ordinary POST bodies.
2. The server is **stateless**. There is no `initialize` handshake to complete and
   no `Mcp-Session-Id` to carry forward, so a bare POST with the bearer token works.

This matters because it is the *only* programmatic path: the REST surface recorded
in the setup notes, `/v1/options/tool/<name>`, returns 404 as of 2026-09-22, as do
six other spellings, and `openapi.json` is 403.

`client.py` wraps it. The key resolves from `QUANTDATA_API_KEY`, falling back to the
bearer token in the MCP registration in `~/.claude.json` — the same
environment-then-MCP-config order `backtest/config.py` uses for Alpaca.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd() / "research" / "quantdata"))
from client import QuantData  # noqa: E402

pd.set_option("display.width", 160, "display.max_columns", 50)

qd = QuantData()
qd.ping()

'QuantData MCP server v1.0.0 — alive.'

### The whole protocol, if you ever need it without the wrapper

```python
POST https://api.quantdata.us/mcp
Authorization: Bearer qd_...
Content-Type:  application/json
Accept:        application/json, text/event-stream

{"jsonrpc": "2.0", "id": 1, "method": "tools/call",
 "params": {"name": "qd_get_order_flow_consolidated",
            "arguments": {"sessionDate": "2026-09-21", "size": 5}}}
```

The reply is `result.content[0].text`: a JSON **string** that holds CSV. Two layers
of decoding, which is why `client.call` unwraps it. Input-validation failures come
back as **HTTP 200** with the complaint in that same text field rather than as a
JSON-RPC error, so the client raises them instead of handing back something that
looks like data.

## 2. The tool catalog

31 tools. They fall into six families, and only the first is the sweep signal —
the rest is positioning, volatility and reference data the strategy does not
currently touch.

In [2]:
FAMILIES = {
    "order flow (the signal)": ["order_flow_consolidated", "order_flow_unconsolidated",
                                "contract_statistics", "contract_trade_side_statistics",
                                "market_share", "gainers_losers"],
    "flow over time":          ["net_drift", "net_flow", "interval_map", "heat_map",
                                "market_map"],
    "positioning / exposure":  ["exposure_by_strike", "exposure_by_expiration",
                                "open_interest_by_strike", "open_interest_by_expiration",
                                "open_interest_over_time", "open_interest_change",
                                "max_pain", "max_pain_over_time"],
    "volatility":              ["iv_rank", "volatility_skew", "volatility_drift",
                                "term_structure"],
    "price series":            ["stock_price_over_time", "option_price_over_time"],
    "equity / dark pool":      ["dark_flow", "dark_pool_levels", "equity_prints"],
    "reference":               ["news_articles", "exchange_notifications", "ping"],
}

tools = {t["name"]: t for t in qd.tools()}
placed = {f"qd_{'get_' if s != 'ping' else ''}{s}" for group in FAMILIES.values() for s in group}
assert placed == set(tools), f"unclassified: {set(tools) ^ placed}"

rows = []
for family, stems in FAMILIES.items():
    for stem in stems:
        name = f"qd_{'get_' if stem != 'ping' else ''}{stem}"
        schema = tools[name].get("inputSchema", {})
        rows.append({
            "family": family,
            "tool": name,
            "params": len(schema.get("properties", {})),
            "summary": tools[name]["description"].strip().split(chr(10))[0][:68],
        })

catalog = pd.DataFrame(rows)
print(f"{len(catalog)} tools across {catalog['family'].nunique()} families")
catalog

31 tools across 7 families


,family,tool,params,summary
0,order flow (the signal),qd_get_order_flow_consolidated,39,Paginated list of CONSOLIDATED option trades —...
1,order flow (the signal),qd_get_order_flow_unconsolidated,39,Paginated list of individual unconsolidated op...
2,order flow (the signal),qd_get_contract_statistics,31,Aggregate option-trade premium / trade count /...
3,order flow (the signal),qd_get_contract_trade_side_statistics,32,"Aggregate one scalar (PREMIUM, TRADE_COUNT, or..."
4,order flow (the signal),qd_get_market_share,31,"Per-exchange rollup of option-trade premium, t..."
5,order flow (the signal),qd_get_gainers_losers,11,Per-ticker rollup of option-trade premium and ...
6,flow over time,qd_get_net_drift,11,Per-bucket net call vs put premium and volume ...
7,flow over time,qd_get_net_flow,13,Per-bucket total call and put activity over ti...
8,flow over time,qd_get_interval_map,10,Time-series of greek-exposure values bucketed ...
9,flow over time,qd_get_heat_map,9,Strike × expiration grid of one option metric ...


Two things worth noting from the parameter counts.

The order-flow tools take **39 parameters** — the entire record is filterable
server-side, including a `filterExpression` for composite predicates. Anything the
sweep screen needs can be pushed down rather than filtered in pandas after the fact.

Everything ending `_over_time`, `_by_strike` or `_by_expiration` takes **two or three**.
Those are per-ticker, per-session snapshots: a range query means a loop over sessions,
one call per session. That shapes any backfill built on them.

## 3. The record

`qd_get_order_flow_consolidated` returns 48 columns per print — the widest record
the API serves, and the one the sweep log is built from. Omit `includes` and every
field populates; pass `includes` and the rest come back as empty strings.

In [3]:
session = "2026-09-21"

flow, meta = qd.frame(
    "qd_get_order_flow_consolidated",
    sessionDate=session,
    sortField="PREMIUM",
    sortDirection="DESCENDING",
    size=50,
)
print(f"{flow.shape[0]} prints x {flow.shape[1]} fields   cursor={meta.get('nextSearchAfter')}")
print()
print("populated on every row:")
print(sorted(flow.columns[flow.notna().all()].tolist()))
print()
print("partially populated:")
print({c: int(flow[c].notna().sum()) for c in flow.columns[flow.notna().any() & flow.isna().any()]})

50 prints x 49 fields   cursor=2588750000,1790011141384

populated on every row:
['askPrice', 'bidAskSpread', 'bidPrice', 'contractType', 'dte', 'exchange', 'expirationDate', 'greeks.charm', 'greeks.color', 'greeks.delta', 'greeks.gamma', 'greeks.omega', 'greeks.rho', 'greeks.sigma', 'greeks.speed', 'greeks.theta', 'greeks.ultima', 'greeks.vanna', 'greeks.vega', 'greeks.veta', 'greeks.vomma', 'greeks.zomma', 'id', 'impliedVolatility', 'industry', 'isETF', 'isGoldenSweep', 'isIndex', 'isOpeningPosition', 'isUnusual', 'isVolumeGreaterThanOpenInterest', 'moneyness.degree', 'moneyness.degreeInPercent', 'moneyness.moneyType', 'openInterest', 'optionPrice', 'premium', 'sector', 'sentimentType', 'size', 'stockPrice', 'strikePrice', 'ticker', 'tradeConsolidationType', 'tradeSideCode', 'tradeTime', 'tradeType', 'ts', 'volume']

partially populated:
{}


All fifteen greeks are served per print — not just delta, but `charm`, `vanna`,
`vomma`, `speed`, `zomma`. Second-order greeks on a single trade are of limited use,
but `delta`, `gamma`, `theta`, `vega` and `impliedVolatility` are struck at the print
and are the vendor values `gates.py` prefers over anything re-derived from bars.

**`bidPrice` / `askPrice` / `bidAskSpread` are populated on every print.** That is
the NBBO *at the moment of the trade* and it is the only quote data the API gives —
see §7, because it is the pivot of the execution question.

In [4]:
cols = ["ts", "ticker", "contractType", "strikePrice", "expirationDate", "stockPrice",
        "optionPrice", "bidPrice", "askPrice", "bidAskSpread", "size", "premium",
        "tradeSideCode", "sentimentType", "tradeConsolidationType", "tradeType"]
flow[cols].head(8)

,ts,ticker,contractType,strikePrice,expirationDate,stockPrice,optionPrice,bidPrice,askPrice,bidAskSpread,size,premium,tradeSideCode,sentimentType,tradeConsolidationType,tradeType
0,2026-09-21 14:57:32.336000-04:00,SPX,CALL,7000.0,2031-12-19,7774.14,2649.42,2498.8,2798.8,300.0,4000,1.059768e+09,ASK,BULLISH,SPLIT,MULTI_FLR_PP
1,2026-09-21 14:57:32.516000-04:00,SPX,CALL,8000.0,2031-12-19,7774.14,2112.08,1962.7,2262.7,300.0,4000,8.448320e+08,BID,BEARISH,SPLIT,MULTI_FLR_PP
2,2026-09-21 11:50:32.049000-04:00,SPX,CALL,7000.0,2027-12-17,7738.86,1332.25,1329.4,1333.5,4.1,5000,6.661250e+08,ASK,BULLISH,SPLIT,M2S_FLR
3,2026-09-21 10:22:13.534000-04:00,SPX,CALL,7000.0,2027-09-17,7719.24,1203.77,1204.1,1207.4,3.3,5000,6.018850e+08,BELOW_BID,BEARISH,BLOCK,M2S_FLR
4,2026-09-21 11:50:32.157000-04:00,SPX,CALL,8000.0,2027-12-17,7738.86,647.95,646.7,649.1,2.4,5000,3.239750e+08,ASK,BULLISH,SPLIT,M2S_FLR
5,2026-09-21 14:57:32.336000-04:00,SPX,PUT,8000.0,2031-12-19,7774.14,747.59,604.5,889.7,285.2,4000,2.990360e+08,ASK,BEARISH,SPLIT,MULTI_FLR_PP
6,2026-09-21 10:22:13.470000-04:00,SPX,CALL,8000.0,2027-09-17,7719.24,511.24,512.2,513.6,1.4,5000,2.556200e+08,BELOW_BID,BEARISH,BLOCK,M2S_FLR
7,2026-09-21 11:50:32.049000-04:00,SPX,PUT,8000.0,2027-12-17,7738.86,509.28,508.5,510.4,1.9,5000,2.546400e+08,BID,BULLISH,SPLIT,M2S_FLR


## 4. The signal — golden sweeps

The screen `build_log.py` transcribes by hand is `tradeTypes=["ISO"]` +
`tradeConsolidationTypes=["SWEEP"]`. ISO is a genuine intermarket sweep; `AUTO` is an
ordinary electronic fill, and anything carrying `COB`, `SPRD` or `TIED` is multi-leg
and directionally ambiguous. The $1M premium floor matches the log's own smallest
entry ($1.03M).

Note `isGoldenSweep` — the vendor computes this flag itself, so the screen can be
expressed either way.

In [4]:
SWEEP_SCREEN = dict(
    tradeConsolidationTypes=["SWEEP"],
    tradeTypes=["ISO"],
    premiumRange={"min": 1_000_000},
    sortField="PREMIUM",
    sortDirection="DESCENDING",
)

sweeps, _ = qd.frame("qd_get_order_flow_consolidated", sessionDate=session, size=100, **SWEEP_SCREEN)

print(f"{session}: {len(sweeps)} ISO sweeps >= $1M   "
      f"({sweeps['premium'].sum() / 1e6:.1f}M total, isGoldenSweep on "
      f"{int(sweeps['isGoldenSweep'].sum())})")
sweeps[["ts", "ticker", "contractType", "strikePrice", "expirationDate", "dte",
        "stockPrice", "optionPrice", "size", "premium", "impliedVolatility",
        "greeks.delta", "volume", "openInterest", "tradeSideCode"]]

2026-09-21: 13 ISO sweeps >= $1M   (31.7M total, isGoldenSweep on 9)


,ts,ticker,contractType,strikePrice,expirationDate,dte,stockPrice,optionPrice,size,premium,impliedVolatility,greeks.delta,volume,openInterest,tradeSideCode
0,2026-09-21 11:05:21.480000-04:00,HUT,CALL,105.0,2027-01-15,116.287940,102.90,21.49,4650,9994876.0,94.652176,0.602263,4651,552,ABOVE_ASK
1,2026-09-21 13:34:04.001000-04:00,SNDK,PUT,1900.0,2026-11-20,60.184664,1743.21,303.00,127,3848140.0,77.104568,-0.536693,202,60,BID
2,2026-09-21 12:10:25-04:00,QQQ,CALL,750.0,2026-10-16,25.201100,738.16,8.74,3001,2624355.0,16.450882,0.395217,16721,38453,ABOVE_ASK
3,2026-09-21 10:29:45.001000-04:00,SPCX,CALL,200.0,2026-12-18,88.312662,157.92,5.39,3860,2083888.0,54.883957,0.243114,4998,24146,ASK
4,2026-09-21 10:14:49.142000-04:00,GOOGL,CALL,365.0,2027-12-17,452.323032,355.60,61.09,284,1735239.0,35.429001,0.613322,320,4891,ASK
5,2026-09-21 15:57:28.698000-04:00,TSLA,PUT,380.0,2026-09-21,0.043414,375.11,4.45,3898,1734940.0,0.047684,-1.000000,24718,198,BELOW_BID
6,2026-09-21 13:16:08.472000-04:00,META,PUT,680.0,2026-10-30,39.155451,733.65,22.00,743,1634632.0,48.875809,-0.278432,830,119,BID
7,2026-09-21 13:29:40.494000-04:00,AAPL,PUT,340.0,2026-09-23,2.146053,338.77,3.05,4999,1524695.0,23.508072,-0.569595,7712,1490,BID
8,2026-09-21 09:38:23.708000-04:00,SNDK,PUT,1705.0,2026-10-16,25.306667,1809.13,84.82,170,1442070.0,72.145462,-0.335369,171,0,BID
9,2026-09-21 13:51:34.386000-04:00,MSFT,PUT,497.5,2026-09-23,2.130845,498.37,3.40,3899,1325660.0,25.606155,-0.454686,10646,158,BID


Every column `build_log.py`'s `SWEEPS` tuples carry by hand is in that frame:
`stockPrice`→`spot`, `optionPrice`→`price`, `impliedVolatility`→`vendor_iv`,
`greeks.delta`→`vendor_delta`, `volume`→`contract_volume`, `openInterest`→`contract_oi`,
plus `industry`. The transcription step is now mechanical.

### The field that reframes the trigger

`sentimentType` looks like vendor intelligence. It is not — it is a **deterministic
function of `contractType` x `tradeSideCode`**, with zero exceptions across every
print checked. That makes it redundant as a feature, and load-bearing as a
definition.

In [6]:
sides, _ = qd.frame("qd_get_order_flow_consolidated", sessionDate=session, size=100,
                    sortField="PREMIUM", sortDirection="DESCENDING")

pd.crosstab([sides["contractType"], sides["tradeSideCode"]], sides["sentimentType"])

sentimentType               BEARISH  BULLISH  NEUTRAL
contractType tradeSideCode                           
CALL         ABOVE_ASK            0        5        0
             ASK                  0       25        0
             BELOW_BID           11        0        0
             BID                 20        0        0
             MID_MARKET           0        0        2
PUT          ABOVE_ASK            5        0        0
             ASK                 13        0        0
             BELOW_BID            0        3        0
             BID                  0       14        0
             MID_MARKET           0        0        2

Read the mapping and the logic is plain: **buying** calls or **selling** puts is
bullish; **selling** calls or **buying** puts is bearish. `ABOVE_ASK` and `ASK` are
aggressive buyers; `BID` and `BELOW_BID` are aggressive sellers.

Which puts a problem under §1 of `strategy.md`:

> | Direction = the sweep's call/put | **assumed** — untested; the whole strategy rests on it |

A put sweep is not bearish. A put sweep **lifted at the offer** is bearish; a put
sweep **hit on the bid** is someone *selling* puts, which is bullish. Direction is
`contractType` x `tradeSideCode`, and the call/put half alone is a coin flip on the
other half. How often does that matter?

In [7]:
screen_days = ["2026-09-21", "2026-09-18", "2026-09-17", "2026-05-13", "2026-01-05", "2025-09-22"]

population = []
for day in screen_days:
    page, _ = qd.frame("qd_get_order_flow_consolidated", sessionDate=day, size=100, **SWEEP_SCREEN)
    page["session"] = day
    population.append(page)
population = pd.concat(population, ignore_index=True)

# what the current rule infers, vs what side the trade actually printed on
population["rule_says"] = population["contractType"].map({"CALL": "BULLISH", "PUT": "BEARISH"})
wrong = population["rule_says"] != population["sentimentType"]

print(f"{len(population)} qualifying ISO sweeps across {len(screen_days)} sessions")
print(f"call/put alone gives the WRONG direction on {wrong.sum()} of them ({wrong.mean():.0%})")
print(f"carrying ${population.loc[wrong, 'premium'].sum() / 1e6:.0f}M of "
      f"${population['premium'].sum() / 1e6:.0f}M total premium")
print()
pd.crosstab(population["contractType"], population["tradeSideCode"])

89 qualifying ISO sweeps across 6 sessions
call/put alone gives the WRONG direction on 52 of them (58%)
carrying $112M of $194M total premium



tradeSideCode,ABOVE_ASK,ASK,BELOW_BID,BID,MID_MARKET
contractType,,,,,
CALL,4,16,8,26,0
PUT,3,14,6,11,1


**58% of the signal population is sold, not bought.** Thirty-four of these sweeps are
calls hit on the bid — someone writing calls, which the current rule books as
bullish. Seventeen are puts hit on the bid — someone writing puts, booked as bearish.

This is not a data-quality caveat, it is a finding about the trigger. §1 grades the
direction rule *"assumed — untested; the whole strategy rests on it."* It is now
tested, and for the majority of signals it is inverted.

Two consequences for the log:

1. `build_log.py`'s `SWEEPS` tuples **do not carry `tradeSideCode`**. The 38 existing
   rows cannot be re-signed from what was transcribed — they have to be re-pulled.
2. Whether direction should be read as aggressor side, or whether the screen should
   simply require `ASK`/`ABOVE_ASK` and drop the sold half, is an open question. Both
   are now testable against ~3,000 prints instead of arguable.

## 5. How far back it goes

The setup note recorded roughly four months of history. It is deeper than that.

In [8]:
probes = ["2025-09-22", "2025-12-01", "2026-01-05", "2026-05-13", "2026-09-21"]

depth = []
for day in probes:
    try:
        page, _ = qd.frame("qd_get_order_flow_consolidated", sessionDate=day, size=100, **SWEEP_SCREEN)
        depth.append({"session": day, "sweeps >= $1M": len(page),
                      "premium $M": round(page["premium"].sum() / 1e6, 1),
                      "top": page.iloc[0]["ticker"] if len(page) else "-"})
    except Exception as exc:  # noqa: BLE001
        depth.append({"session": day, "sweeps >= $1M": f"ERROR {exc}"[:40]})

pd.DataFrame(depth)

,session,sweeps >= $1M,premium $M,top
0,2025-09-22,18,39.2,SPY
1,2025-12-01,9,12.3,CVNA
2,2026-01-05,11,20.4,NFLX
3,2026-05-13,23,39.3,COHR
4,2026-09-21,13,31.7,HUT


**At least twelve months, and the daily count is stable.** That retires the standing
caveat in `research/sweep_events/README.md` — *"neither Alpaca nor Robinhood serves
historical sweep prints, so the log only grows forward"*. At roughly a dozen
qualifying sweeps a session, a year is on the order of **3,000 prints** against the
38 hand-transcribed ones the strategy currently rests on.

The sample-size problem is now a data-engineering problem, which is a much better
problem to have.

## 6. Historical option prices — the capability that changes the backtest

`qd_get_option_price_over_time` returns OHLC + volume for **one specific option
contract**, on any past session. This is the piece Alpaca's lane does not serve
well, and the backtest currently substitutes with simulation.

Below: the contract the strategy would actually have bought after the TSLA sweep on
2026-09-14 (spot 360.34 → ATM strike 360, third-Friday monthly → 2026-10-16),
tracked across the sessions that followed.

In [9]:
def contract_bars(ticker, expiry, strike, right, day, period="ONE_HOUR"):
    bars, _ = qd.frame(
        "qd_get_option_price_over_time",
        ticker=ticker, expirationDate=expiry, strikePrice=strike,
        contractType=right, sessionDate=day, aggregationPeriod=period,
    )
    return bars

path = []
for day in ["2026-09-14", "2026-09-15", "2026-09-16", "2026-09-17", "2026-09-18", "2026-09-21"]:
    bars = contract_bars("TSLA", "2026-10-16", 360.0, "CALL", day)
    path.append({"session": day, "bars": len(bars),
                 "open": bars["openPrice"].iloc[0], "close": bars["closePrice"].iloc[-1],
                 "low": bars["lowPrice"].min(), "high": bars["highPrice"].max(),
                 "volume": int(bars["volume"].sum())})

tsla = pd.DataFrame(path)
tsla["ret vs entry %"] = (tsla["close"] / tsla["close"].iloc[0] - 1).mul(100).round(1)
tsla

,session,bars,open,close,low,high,volume,ret vs entry %
0,2026-09-14,7,18.50,17.34,16.48,22.51,2676,0.0
1,2026-09-15,7,16.70,16.18,15.14,19.36,3129,-6.7
2,2026-09-16,7,16.57,17.01,15.70,20.90,2973,-1.9
3,2026-09-17,7,22.00,20.64,19.80,26.55,1380,19.0
4,2026-09-18,7,22.35,18.99,17.60,23.30,1727,9.5
5,2026-09-21,7,22.00,25.15,22.00,27.75,961,45.0


Real traded prices, real volume, hour by hour — no Black-Scholes assumption anywhere.

### The caveat that comes with it

These are **trade** bars, not **quote** bars. A bar exists only where the contract
actually traded, so the series is irregular and its density is a liquidity measure in
its own right. Compare the strategy's ATM monthly against the contract the
institution swept in the META print of 2026-09-09:

In [10]:
liquidity = []
for label, args in {
    "TSLA 10/16 360C  (ATM monthly — what the strategy buys)":
        ("TSLA", "2026-10-16", 360.0, "CALL", "2026-09-18"),
    "META 10/16 655C  (the swept contract itself)":
        ("META", "2026-10-16", 655.0, "CALL", "2026-09-18"),
}.items():
    bars = contract_bars(*args)
    liquidity.append({
        "contract": label,
        "bars in session": len(bars),
        "contracts traded": int(bars["volume"].sum()),
        "median bar volume": int(bars["volume"].median()),
    })

pd.DataFrame(liquidity)

,contract,bars in session,contracts traded,median bar volume
0,TSLA 10/16 360C (ATM monthly — what the strat...,7,1727,255
1,META 10/16 655C (the swept contract itself),6,22,3


Three orders of magnitude apart. The ATM monthly trades in the thousands per session
— an exit at the marked price is plausible. The swept contract trades a handful of
contracts an hour, and marking an exit against a 2-lot print is fiction.

This is a **result**, not a caveat to note and move past: §3 of `strategy.md` argues
on return grounds that the strategy should not buy the swept contract. The liquidity
data says the swept contract could not have been exited at the marked price either
way. Two independent arguments for the same rule.

Any backtest built on this feed has to carry the volume column through and refuse to
mark an exit into a bar that did not trade enough to absorb the position.

## 7. What the feed does *not* serve

Worth being precise, because these are the boundaries of anything built on it.

In [11]:
boundaries = pd.DataFrame([
    ("continuous NBBO", "bid/ask AT each print only",
     "no quote at an arbitrary moment; exit spread unobservable"),
    ("earnings calendar", "news_articles only, no scheduled dates",
     "the earnings veto stays hand-maintained in build_log.py"),
    ("splits / corporate actions", "not served; spot is unadjusted",
     "must reconcile against Alpaca's adjusted bars — the CRWD +5,368% bug"),
    ("point-in-time open interest", "OI is the T+1 published figure",
     "same-session OI on a print is mildly forward-looking"),
    ("order identity", "no participant or account linkage",
     "cannot follow one institution's position through time"),
    ("fills / executions", "read-only market data",
     "Robinhood MCP remains the only execution path"),
], columns=["not served", "what exists instead", "consequence"])
boundaries

,not served,what exists instead,consequence
0,continuous NBBO,bid/ask AT each print only,no quote at an arbitrary moment; exit spread u...
1,earnings calendar,"news_articles only, no scheduled dates",the earnings veto stays hand-maintained in bui...
2,splits / corporate actions,not served; spot is unadjusted,must reconcile against Alpaca's adjusted bars ...
3,point-in-time open interest,OI is the T+1 published figure,same-session OI on a print is mildly forward-l...
4,order identity,no participant or account linkage,cannot follow one institution's position throu...
5,fills / executions,read-only market data,Robinhood MCP remains the only execution path


## 8. Where this leaves the system

Three backends, and with historical option prices the **data** loop does close:

| Need | Backend | Status |
|---|---|---|
| Sweep signal, historical | Quant Data | **new** — 12 months, ~3,000 prints |
| Signal *direction* | Quant Data `tradeSideCode` | **new** — and it contradicts the current rule |
| Underlying bars | Alpaca | in place, split-adjusted |
| Option prices, historical | Quant Data | **new** — replaces simulation |
| Live quotes | Robinhood MCP | in place |
| Execution | Robinhood MCP | in place, `execution/PROTOCOL.md` |
| Earnings dates | *hand-maintained* | **gap** |
| Corporate actions | Alpaca `get_corporate_actions` | **unwired** |

What does *not* close is the **validation** loop. The gaps below are ordered by how
much each one is currently distorting the reported number.

In [12]:
gaps = pd.DataFrame([
    (1, "Direction rule is inverted for most signals",
     "the trigger reads direction from call/put alone; 58% of qualifying sweeps print "
     "on the bid and are sold, not bought. The log does not even record the side.",
     "CLOSABLE NOW — tradeSideCode ships on every print"),
    (2, "Simulated option prices",
     "backtest_strategy.price_path() prices every entry and exit with Black-Scholes "
     "off the underlying path at constant IV. No traded price is consulted.",
     "CLOSABLE NOW — qd_get_option_price_over_time, one call per contract-session"),
    (3, "Selection bias in the log",
     "38 sweeps chosen by hand from a dashboard. Rejects were never logged, so no gate "
     "can be shown to add edge rather than cut winners.",
     "CLOSABLE NOW — pull every qualifying sweep, kept and rejected alike"),
    (4, "No execution model",
     "entries assume a fill at the vendor's print price; exits assume a fill at a bar "
     "close. Neither pays a spread.",
     "PARTIAL — bidAskSpread per print gives an empirical distribution to charge; "
     "no continuous NBBO to verify against"),
    (5, "No capacity constraint",
     "position size is never checked against what the contract actually traded.",
     "CLOSABLE NOW — volume rides along in the option bars"),
    (6, "Hand-maintained earnings veto",
     "a symbol missing from EARNINGS silently passes the veto — the log's most likely "
     "corruption path.",
     "NEEDS A SOURCE — Robinhood get_earnings_calendar, or a vendor"),
    (7, "Unreconciled corporate actions",
     "vendor spot is unadjusted, Alpaca bars are adjusted; gates.split_factor() "
     "hand-corrects recognisable splits.",
     "CLOSABLE — wire Alpaca get_corporate_actions into the reconciliation"),
    (8, "Multiple testing",
     "nine gates scored against 38 observations. At n=3,000 the same nine gates become "
     "testable; at n=38 they are not.",
     "FOLLOWS FROM #3 — plus an out-of-sample split held back from the start"),
], columns=["#", "gap", "what is wrong today", "path"])

pd.set_option("display.max_colwidth", 96)
gaps.set_index("#")

,gap,what is wrong today,path
#,,,
1,Direction rule is inverted for most signals,the trigger reads direction from call/put alone; 58% of qualifying sweeps print on the bid a...,CLOSABLE NOW — tradeSideCode ships on every print
2,Simulated option prices,backtest_strategy.price_path() prices every entry and exit with Black-Scholes off the underl...,"CLOSABLE NOW — qd_get_option_price_over_time, one call per contract-session"
3,Selection bias in the log,"38 sweeps chosen by hand from a dashboard. Rejects were never logged, so no gate can be show...","CLOSABLE NOW — pull every qualifying sweep, kept and rejected alike"
4,No execution model,entries assume a fill at the vendor's print price; exits assume a fill at a bar close. Neith...,PARTIAL — bidAskSpread per print gives an empirical distribution to charge; no continuous NB...
5,No capacity constraint,position size is never checked against what the contract actually traded.,CLOSABLE NOW — volume rides along in the option bars
6,Hand-maintained earnings veto,a symbol missing from EARNINGS silently passes the veto — the log's most likely corruption p...,"NEEDS A SOURCE — Robinhood get_earnings_calendar, or a vendor"
7,Unreconciled corporate actions,"vendor spot is unadjusted, Alpaca bars are adjusted; gates.split_factor() hand-corrects reco...",CLOSABLE — wire Alpaca get_corporate_actions into the reconciliation
8,Multiple testing,"nine gates scored against 38 observations. At n=3,000 the same nine gates become testable; a...",FOLLOWS FROM #3 — plus an out-of-sample split held back from the start


### The short answer

**Signal, history and execution are covered. Verification is not — and one of the
gaps is not a verification problem at all.**

Gap #1 is a live defect in the trigger, and it is the one to fix first. Everything
downstream — gates, contract choice, sizing — is conditioned on a direction that is
inverted for 58% of the signal population. Backtesting the rest more accurately while
the sign is wrong just measures the error more precisely.

After that, #2 and #3 are the ones distorting the reported number: every return in
the current backtest is a Black-Scholes price of a hand-picked sweep, and neither of
those words belongs in a result anyone trades on.

The two gaps that do *not* close from this feed are **earnings dates** and a
**verifiable execution model**. Earnings needs a source, wherever it comes from. The
execution gap is structural: with prints but no continuous quotes, a fill assumption
can be made *empirical* — charge the observed spread distribution — but never
*verifiable*. Only forward paper trading through Robinhood closes that one, which is
why the forward log keeps its value even now that history is available.

Suggested order: **#1, then #3, then #2.** Fix the sign, pull the full population,
then price it with real option bars.